# Is the uncertainty correctly uncertain, or blindly confident?

24 nanobodies SimpleFold got **badly wrong** in the benchmark (>6 A) and 12 it got **right** (<1.5 A).
None were used to calibrate anything. Fold 6 samples each, measure spread, and ask whether the
hard ones are flagged.

The dangerous failure mode is **high error with low spread** -- confidently wrong. That is what
this counts.

Error is re-scored in this run, because SimpleFold's per-target error is not reproducible
across runs (median 0.40 A, max 2.12 A drift).

In [ ]:
import os, sys, json, time, glob, subprocess, shutil
import numpy as np
PICKS = ['7A4TA', '7BNWB', '7KD0C', '7MFVB', '7MJHF', '7MY3D', '7N0RC', '7N9TE', '7OMNC', '7QVKBBB', '7ZMVF', '8DQUA', '8ELNH', '8ELPB', '8ELQB', '8EN4C', '8GHRD', '8H5UB', '8OWWB', '8S5WB', '8V9XA', '8XPSB', '8YSFD', '9EMYK', '9ETLD', '9G3EC', '9GV4A', '9HO4C', '9HURB', '9LR1C', '9MTEB', '9MY8A', '9P6XF', '9Q6NC', '9W3LB', '9Z5QA']
HARD = ['9HO4C', '9MY8A', '7QVKBBB', '7N9TE', '7BNWB', '7MJHF', '9P6XF', '9LR1C', '8ELPB', '8GHRD', '8ELNH', '9ETLD', '9W3LB', '9Z5QA', '8XPSB', '7ZMVF', '7A4TA', '8V9XA', '8S5WB', '9EMYK', '8ELQB', '7MY3D', '7OMNC', '7N0RC']
NS, STEPS, TAU, SEED = '6','500','0.01','12345'
WORK, SCRATCH = '/kaggle/working','/tmp/sf'
os.makedirs(SCRATCH, exist_ok=True)
R={'n_samples':int(NS),'note':'24 known-hard + 12 known-easy; is spread correctly uncertain?'}
import torch
assert torch.cuda.device_count()==2 and torch.cuda.get_device_capability(0)>=(7,0)
R['gpus']=[torch.cuda.get_device_name(g) for g in range(2)]
_ds=glob.glob('/kaggle/input/**/targets_vhh.json',recursive=True); assert len(_ds)==1
TG={f"{t['pdb']}{t['H']}": t for t in json.load(open(_ds[0]))}
TG={k:TG[k] for k in PICKS}; assert len(TG)==36
print(R['gpus'], len(TG),'targets')
def must(c,w):
    p=subprocess.run(c,shell=True,text=True,capture_output=True)
    if p.returncode: raise RuntimeError(w+':\n'+(p.stdout+p.stderr)[-1500:])
must('git clone -q --depth 1 -b fix-multichain-fasta-parsing '
     'https://github.com/omar-A-hassan/ml-simplefold '+SCRATCH+'/repo','clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo','install')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git','esm')
must(f'{sys.executable} -m pip install -q biotite','biotite')
INF=f'{SCRATCH}/repo/src/simplefold/inference.py'; s=open(INF).read()
a='        esm_model = esm_model.to(device)'; assert s.count(a)==1
open(INF,'w').write(s.replace(a,'        esm_model = esm_model.half().to(device)'))
print('fp16 patched')

## Fold

In [ ]:
fadir=f'{SCRATCH}/fa'; os.makedirs(fadir,exist_ok=True)
for tid,t in TG.items(): open(f'{fadir}/{tid}.fasta','w').write('>A|'+tid+'\n'+t['hseq']+'\n')
w=subprocess.run([sys.executable,'-c',"import torch,esm; torch.hub.load('facebookresearch/esm:main','esm2_t36_3B_UR50D')"],capture_output=True,text=True)
if w.returncode: raise RuntimeError((w.stdout+w.stderr)[-1500:])
tids=sorted(TG)
for g in (0,1):
    d=f'{SCRATCH}/sh{g}'; shutil.rmtree(d,ignore_errors=True); os.makedirs(d)
for i,tid in enumerate(tids): os.link(f'{fadir}/{tid}.fasta', f'{SCRATCH}/sh{i%2}/{tid}.fasta')
t0=time.time(); procs=[]
for g in (0,1):
    if g: time.sleep(60)
    procs.append(subprocess.Popen(['simplefold','--simplefold_model','simplefold_100M','--backend','torch',
        '--ckpt_dir',f'{SCRATCH}/artifacts','--num_steps',STEPS,'--tau',TAU,'--nsample_per_protein',NS,
        '--seed',SEED,'--fasta_path',f'{SCRATCH}/sh{g}','--output_dir',f'{SCRATCH}/out{g}'],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
        env={**os.environ,'CUDA_VISIBLE_DEVICES':str(g)}))
    print('launched',g,flush=True)
for g,p in enumerate(procs):
    o=p.communicate()[0]; R.setdefault('rc',{})[g]=p.returncode
    if p.returncode: print('GPU'+str(g)+' FAILED',o[-2000:])
R['fold_min']=round((time.time()-t0)/60,1); R['n_cif']=len(glob.glob(f'{SCRATCH}/out*/**/*.cif',recursive=True))
print(R['n_cif'],'structures in',R['fold_min'],'min')

## Score + spread

In [ ]:
import biotite.structure.io.pdbx as pdbx
def kabsch(P,Q):
    Pm,Qm=P.mean(0),Q.mean(0)
    V,_,W=np.linalg.svd((P-Pm).T@(Q-Qm)); D=np.diag([1,1,np.sign(np.linalg.det(V@W))])
    return V@D@W,Pm,Qm
rmsd=lambda a,b: float(np.sqrt(((a-b)**2).sum(-1).mean()))
AT=['N','CA','C','CB']; rows=[]
for tid,t in TG.items():
    g=sorted(glob.glob(f'{SCRATCH}/out*/**/*'+tid+'*.cif',recursive=True))
    if len(g)<2: R.setdefault('skip',[]).append(tid); continue
    Ln=len(t['hseq']); lo,hi=t['cdr3']; c3=np.zeros(Ln,bool); c3[lo:hi]=True
    fwm=np.ones(Ln,bool)
    for k in ('cdr1','cdr2','cdr3'):
        a,b=t[k]; fwm[a:b]=False
    S=[]
    for fp in g:
        a=pdbx.get_structure(pdbx.CIFFile.read(fp),model=1); u=np.unique(a.res_id)
        if len(np.unique(a.chain_id))!=1 or len(u)!=Ln: break
        P=np.full((Ln,4,3),np.nan); rid=a.res_id-a.res_id.min()
        for k,an in enumerate(AT):
            mm=a.atom_name==an; P[rid[mm],k]=a.coord[mm]
        S.append(P)
    if len(S)<2: R.setdefault('skip',[]).append(tid+':parse'); continue
    ref=np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']],float)
    def err(P):
        ok=np.isfinite(ref[...,0])&np.isfinite(P[...,0]); sel=ok&fwm[:,None]
        if sel.sum()<60: return None
        Rot,Pm,Qm=kabsch(P[sel],ref[sel]); al=np.full_like(P,np.nan); al[ok]=(P[ok]-Pm)@Rot+Qm
        s2=ok&c3[:,None]
        return rmsd(al[s2],ref[s2]) if s2.sum()>=3 else None
    e=[err(P) for P in S]
    if any(x is None for x in e): R.setdefault('skip',[]).append(tid+':score'); continue
    base=S[0]; AL=[base]
    for P in S[1:]:
        m=np.isfinite(P[...,0])&np.isfinite(base[...,0])&fwm[:,None]
        Rot,Pm,Qm=kabsch(P[m],base[m]); AL.append((P-Pm)@Rot+Qm)
    AL=np.stack(AL)
    rows.append({'tid':tid,'group':'hard' if tid in HARD else 'easy',
                 'spread_cdr3':round(float(np.nanmean(np.nanstd(AL,axis=0)[c3])),4),
                 'err':round(float(e[0]),3),'err_mean':round(float(np.mean(e)),3),
                 'err_oracle':round(float(min(e)),3),'cdr3_len':int(hi-lo)})
R['rows']=rows
print(len(rows),'scored; skips',len(R.get('skip',[])))
import numpy as _n
for grp in ('hard','easy'):
    sub=[r for r in rows if r['group']==grp]
    print(f"  {grp}: n={len(sub)} spread median {_n.median([r['spread_cdr3'] for r in sub]):.3f}  "
          f"error median {_n.median([r['err'] for r in sub]):.2f}")
json.dump(R, open(f'{WORK}/hard_results.json','w'), indent=1, default=str)